# Multimodal Strategy — Text + Numerical Embeddings

Runs S1 Hard70 (text) and MOMENT+OpenAI (numerical) side-by-side on the same dataset.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
%matplotlib inline
import sys, os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')
from sentence_transformers import SentenceTransformer
from momentfm import MOMENTPipeline
from openai import OpenAI
from src.backtest.engine import BacktestEngine, BacktestConfig, decompose_alpha_beta
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, RED, WHITE, CYAN, PURPLE
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

## 1. Load Data — 10 tickers with text + prices

In [ ]:
news_dir = Path('data/text')
price_dir = Path('data/time_series')

text_tickers = {f.stem.upper() for f in news_dir.glob('*.jsonl')}
price_tickers = {f.stem.upper() for f in price_dir.glob('*.csv')}
tickers = sorted(text_tickers & price_tickers)[:10]

all_articles = []
for t in tickers:
    fp = news_dir / f'{t}.jsonl'
    with open(fp) as f:
        for line in f:
            try:
                doc = json.loads(line.strip())
                d = doc.get('Date', '')
                txt = doc.get('Article', '') or doc.get('Article_title', '')
                if d and txt and len(txt) > 30:
                    all_articles.append({'ticker': t, 'date': str(d)[:10], 'text': txt[:800]})
            except:
                continue

df_articles = pd.DataFrame(all_articles)
df_articles['date'] = pd.to_datetime(df_articles['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
df_articles = df_articles.dropna(subset=['date'])

pframes = []
for t in tickers:
    ts_file = price_dir / f'{t.lower()}.csv'
    px = pd.read_csv(ts_file, usecols=['Date', 'Close', 'Open', 'High', 'Low', 'Volume'])
    px['date'] = pd.to_datetime(px['Date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
    px = px.dropna(subset=['date', 'Close']).sort_values('date')
    px['ticker'] = t
    pframes.append(px[['date', 'ticker', 'Close', 'Open', 'High', 'Low', 'Volume']])
prices = pd.concat(pframes, ignore_index=True)
px_panel = prices.pivot_table(index='date', columns='ticker', values='Close', aggfunc='last')

pd.DataFrame({
    '': ['tickers', 'articles', 'price_dates', 'common_dates'],
    'value': [len(tickers), len(df_articles), px_panel.shape[0], '...']
})

## 2. Text Embedding Strategy (SentenceTransformer → S1 Hard70)

In [ ]:
daily_text = df_articles.groupby(['ticker', 'date'])['text'].apply(lambda x: ' '.join(x)[:3000]).reset_index()

st_model = SentenceTransformer('all-MiniLM-L6-v2')
text_embs = st_model.encode(daily_text['text'].tolist(), show_progress_bar=False)

dates_all = sorted(daily_text['date'].unique())
text_emb_panel = {}
for t in tickers:
    sub = daily_text[daily_text['ticker'] == t].set_index('date')['text']
    # reindex to all dates
    sub_idx = sub.index.unique()
    text_emb_panel[t] = pd.Series(0, index=dates_all, dtype=float)

print(f'Text embeddings: {text_embs.shape}')

## 2a. Build daily text embedding panel

In [ ]:
emb_dim = text_embs.shape[1]
text_panel_dict = {}
for t in tickers:
    sub = daily_text[daily_text["ticker"] == t].copy()
    sub["_idx"] = sub.index - daily_text.index[0]
    emb_by_date = {}
    for _, row in sub.iterrows():
        d = row["date"]
        idx = int(row["_idx"])
        if idx >= 0 and idx < len(text_embs):
            if d in emb_by_date:
                emb_by_date[d].append(text_embs[idx])
            else:
                emb_by_date[d] = [text_embs[idx]]
    default_emb = np.zeros(emb_dim)
    series_vals = []
    last = default_emb.copy()
    for d in dates_all:
        if d in emb_by_date:
            arrs = emb_by_date[d]
            avg = np.mean(arrs, axis=0)
            series_vals.append(avg)
            last = avg
        else:
            series_vals.append(last.copy())
    text_panel_dict[t] = series_vals

cd_text = sorted(set(dates_all) & set(px_panel.index))
cd_text[:5], len(cd_text)


## 2b. Drift → Intensity → S1 Signal

In [ ]:
# Align
cd = cd_text
emb_dict = {}
for t in tickers:
    date_to_emb = dict(zip(dates_all, text_panel_dict[t]))
    emb_dict[t] = pd.Series([date_to_emb[d] for d in cd], index=cd)
emb_panel = pd.DataFrame({t: [emb_dict[t].iloc[i] for i in range(len(cd))] for t in tickers}, index=cd)

n_dates = len(cd)
n_tickers = len(tickers)

# Embedding drift
drift = np.zeros((n_dates, n_tickers))
for j in range(n_tickers):
    prev = emb_panel.iloc[0, j]
    for i in range(1, n_dates):
        cur = emb_panel.iloc[i, j]
        drift[i, j] = np.linalg.norm(cur - prev)
        prev = cur
drift_df = pd.DataFrame(drift, index=cd, columns=tickers)

# Hawkes intensity
p95 = np.percentile(drift_df.values.flatten(), 95)
events = (drift_df >= p95).astype(float)
intensity = pd.DataFrame(0.0, index=cd, columns=tickers)
for j in range(n_tickers):
    ev = events.iloc[:, j].values
    lam = 0.0
    for i in range(n_dates):
        lam = np.exp(-0.2) * lam + 0.8 * ev[i]
        intensity.iloc[i, j] = lam

# Regime + signal
px = px_panel.loc[cd]
ret_raw = px.pct_change().fillna(0.0)
known_ret = ret_raw.shift(1).fillna(0.0)
ret_df = ret_raw.shift(-1).fillna(0.0)
mask = (~px.isna()).astype(float)

mkt_ret = known_ret.mean(axis=1)
regime = (mkt_ret.rolling(20).std() > mkt_ret.rolling(20).std().median()).astype(float).fillna(0.0)

int_rank = intensity.rank(axis=1, pct=True)
hard70 = (int_rank >= 0.70).astype(float)
reg_mat = pd.DataFrame(np.repeat(regime.values[:, None], n_tickers, axis=1), index=cd, columns=tickers)
rev1 = -np.sign(known_ret).fillna(0.0)
mom3 = np.sign(known_ret.rolling(3, min_periods=2).mean()).fillna(0.0)
raw_text = hard70 * (reg_mat * rev1 + (1.0 - reg_mat) * mom3)

pd.DataFrame({
    'step': ['drift_p95', 'events_frac', 'regime_high_vol_pct', 'gate_active_pct'],
    'value': [f'{p95:.4f}', f'{events.mean().mean()*100:.1f}%', f'{regime.mean()*100:.1f}%', f'{hard70.mean().mean()*100:.1f}%']
})

## 3. Numerical Strategy — MOMENT + OpenAI

In [ ]:
# MOMENT embeddings on price windows
print('Loading MOMENT...')
moment = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'embedding'},
    torch_dtype=torch.float32,
)
moment.eval()

# OpenAI embeddings on news summaries
api_key = os.environ.get('OPENAI_API_KEY', '').strip()
if not api_key:
    with open(_root / '.env') as f:
        for line in f:
            if 'OPENAI_API_KEY' in line:
                api_key = line.split('=', 1)[1].strip().strip('"').strip("'")
                os.environ['OPENAI_API_KEY'] = api_key
                break

openai_client = OpenAI(api_key=api_key) if api_key else None
print(f'MOMENT loaded | OpenAI: {bool(openai_client)}')

## 3a. Price window → MOMENT embeddings

In [ ]:
window = 30
moment_dim = 1280
moment_emb_dict = {t: np.zeros((n_dates, moment_dim)) for t in tickers}

price_cols = ['Close', 'Open', 'High', 'Low', 'Volume']

for j, t in enumerate(tickers):
    px_t = px_panel[t].values
    # Fill NaN with forward fill
    mask_nan = np.isnan(px_t)
    last = px_t[~mask_nan][0] if (~mask_nan).any() else 0
    for i in range(len(px_t)):
        if mask_nan[i]: px_t[i] = last
        else: last = px_t[i]

    for i in range(window, n_dates):
        win = px_t[i - window : i]
        chunk = torch.tensor(win, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
        chunk = chunk.repeat(1, 1, 5)
        with torch.no_grad():
            out = moment(chunk)
            if hasattr(out, 'embeddings'):
                moment_emb_dict[t][i] = out.embeddings.squeeze().numpy()
            else:
                moment_emb_dict[t][i] = out.last_hidden_state.mean(dim=1).squeeze().numpy()

print(f'MOMENT embeddings: {n_dates} dates x {n_tickers} tickers x {moment_dim} dim')

## 3b. News text → OpenAI embeddings

In [ ]:
openai_dim = 1536
openai_emb_dict = {t: np.zeros((n_dates, openai_dim)) for t in tickers}

if openai_client:
    daily_news = daily_text.copy()
    daily_news['text_short'] = daily_news['text'].str[:500]

    for t in tickers:
        sub = daily_news[daily_news['ticker'] == t]
        date_to_text = {}
        for _, row in sub.iterrows():
            d = row['date']
            txt = row['text_short']
            if d in date_to_text:
                date_to_text[d] += ' ' + txt
            else:
                date_to_text[d] = txt

        for i, d in enumerate(cd):
            if d in date_to_text:
                try:
                    resp = openai_client.embeddings.create(
                        model='text-embedding-3-small',
                        input=date_to_text[d][:8000],
                    )
                    openai_emb_dict[t][i] = np.array(resp.data[0].embedding)
                except:
                    pass

    print(f'OpenAI embeddings computed')
else:
    print('OpenAI client not available — using zeros')

## 3c. Fuse MOMENT + OpenAI → Numerical Signal

In [ ]:
# Simple fusion: concatenate + PCA to 64 dims, then regime router
from sklearn.decomposition import PCA

fused_dim = 64
pca = PCA(n_components=fused_dim)

all_fused = []
for j, t in enumerate(tickers):
    m_emb = moment_emb_dict[t]
    o_emb = openai_emb_dict[t]
    fused = np.concatenate([m_emb, o_emb], axis=1)
    all_fused.append(fused)

stacked = np.stack(all_fused, axis=1)

# Flatten to 2D for PCA fit
flat = stacked.reshape(-1, stacked.shape[-1])
flat_clean = flat[~np.isnan(flat).any(axis=1)]
if len(flat_clean) > fused_dim:
    pca.fit(flat_clean)
    fused_reduced = pca.transform(flat).reshape(n_dates, n_tickers, fused_dim)
else:
    fused_reduced = np.zeros((n_dates, n_tickers, fused_dim))

# Drift on fused embeddings
num_drift = np.zeros((n_dates, n_tickers))
for i in range(1, n_dates):
    num_drift[i] = np.linalg.norm(fused_reduced[i] - fused_reduced[i-1], axis=1)
num_drift_df = pd.DataFrame(num_drift, index=cd, columns=tickers)

p95_num = np.percentile(num_drift_df.values.flatten(), 95)
num_events = (num_drift_df >= p95_num).astype(float)
num_intensity = pd.DataFrame(0.0, index=cd, columns=tickers)
for j in range(n_tickers):
    ev = num_events.iloc[:, j].values
    lam = 0.0
    for i in range(n_dates):
        lam = np.exp(-0.2) * lam + 0.8 * ev[i]
        num_intensity.iloc[i, j] = lam

num_int_rank = num_intensity.rank(axis=1, pct=True)
num_hard70 = (num_int_rank >= 0.70).astype(float)
raw_numerical = num_hard70 * (reg_mat * rev1 + (1.0 - reg_mat) * mom3)

pd.DataFrame({
    '': ['MOMENT_dim', 'OpenAI_dim', 'fused_dim', 'fused_drift_p95'],
    'value': [moment_dim, openai_dim, fused_dim, f'{p95_num:.4f}']
})

## 4. Portfolio Construction + Backtest (Both Strategies)

In [ ]:
def run_portfolio(raw_signal, name):
    def tilt_fn(ra):
        x = (ra * mask).fillna(0.0)
        x = x.sub(x.mean(axis=1), axis=0)
        x = x.div(x.abs().sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
        return x

    tilt = tilt_fn(raw_signal)
    bh_w = mask.div(mask.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    bh_ret = (bh_w * ret_df.fillna(0.0)).sum(axis=1)
    mkt_trend = known_ret.mean(axis=1).rolling(20, min_periods=10).mean().fillna(0.0)
    exposure = pd.Series(np.where(mkt_trend > 0, 1.30, 1.00), index=cd)
    w = (bh_w + 0.05 * tilt).clip(lower=0.0)
    w = w.div(w.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    base = (w * ret_df.fillna(0.0)).sum(axis=1)
    sr = exposure * base
    return sr, bh_ret

split_idx = int(n_dates * 0.7)

sr_text, bh_ret = run_portfolio(raw_text, 'Text (S1 Hard70)')
sr_numerical, _ = run_portfolio(raw_numerical, 'Numerical (MOMENT+OpenAI)')

strat_text_test = sr_text.iloc[split_idx:]
bh_test = bh_ret.iloc[split_idx:]
strat_num_test = sr_numerical.iloc[split_idx:]

def stats(r):
    r = r.fillna(0.0)
    t = (1.0 + r).prod() - 1.0
    a = (1.0 + t) ** (252.0 / max(len(r), 1)) - 1.0
    v = r.std(ddof=0) * np.sqrt(252.0)
    return t, a, v, a / v if v > 0 else 0.0

st_t, st_a, st_v, st_sh = stats(strat_text_test)
sn_t, sn_a, sn_v, sn_sh = stats(strat_num_test)
bh_t, bh_a, bh_v, bh_sh = stats(bh_test)

ab_text = decompose_alpha_beta(strat_text_test, bh_test, 252)
ab_num = decompose_alpha_beta(strat_num_test, bh_test, 252)

pd.DataFrame({
    'Strategy': ['Text (S1 Hard70)', 'Numerical (MOMENT+OpenAI)', 'Buy & Hold'],
    'Test Return': [f'{st_t*100:+.1f}%', f'{sn_t*100:+.1f}%', f'{bh_t*100:+.1f}%'],
    'Ann. Return': [f'{st_a*100:.2f}%', f'{sn_a*100:.2f}%', f'{bh_a*100:.2f}%'],
    'Sharpe': [f'{st_sh:.3f}', f'{sn_sh:.3f}', f'{bh_sh:.3f}'],
    'Alpha (ann)': [f'{ab_text["alpha_pct"]:+.2f}%', f'{ab_num["alpha_pct"]:+.2f}%', '—'],
    'Beta': [f'{ab_text["beta"]:.3f}', f'{ab_num["beta"]:.3f}', '—'],
    'R²': [f'{ab_text["r_squared"]:.3f}', f'{ab_num["r_squared"]:.3f}', '—'],
})

## 5. Equity Curves — Both Strategies vs BH

In [ ]:
eq_text = (1.0 + sr_text).cumprod() * 100
eq_num = (1.0 + sr_numerical).cumprod() * 100
eq_bh = (1.0 + bh_ret).cumprod() * 100

images = _root / 'images'
images.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor(BG)
ax.plot(eq_bh.index, eq_bh.values, color=WHITE, lw=1.2, ls='--', alpha=0.5, label=f'Buy & Hold ({bh_t*100:+.1f}%)')
ax.plot(eq_text.index, eq_text.values, color=GOLD, lw=2.5, label=f'Text S1 Hard70 ({st_t*100:+.1f}%)')
ax.plot(eq_num.index, eq_num.values, color=CYAN, lw=2.5, label=f'Numerical MOMENT+OpenAI ({sn_t*100:+.1f}%)')
ax.axvline(cd[split_idx], color=WHITE, ls=':', alpha=0.3, label='Train/Test')
ax.fill_between(eq_bh.index, eq_bh.values, 100, where=eq_bh.values >= 100, color=GREEN, alpha=0.04)
ax.axhline(100, color='white', ls='--', alpha=0.15)
ax.set_title(f'Multimodal Strategies vs Buy & Hold — {n_tickers} tickers, {len(cd)} days', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')
fig.tight_layout()
fig.savefig(images / 'multimodal_strategies_vs_bh.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()